# SMCO AUTO SN

In [6]:

from decimal import Decimal
import locale
from concurrent.futures import ThreadPoolExecutor
import threading
import sys
import os
from xml.dom.minidom import Document
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
import re
import win32com.client as comclt
import time
import pandas as pd
import numpy as np
from tkinter import font
from tkinter import ttk
from tkinter import filedialog
from tkinter import messagebox
from tkinter import *
import traceback
from bs4 import BeautifulSoup
from googletrans import Translator
import requests

opt = Options()

opt.add_experimental_option("debuggerAddress", "localhost:8989")


driver = webdriver.Chrome(service=Service(
    r'C:\bin\chromedriver.exe'), options=opt)

print("handles ",driver.window_handles)



titleList = []
value_list = []

title_dict_sorted = {}
title_dict = {}
title_order = ['Seller Centre', 'SMCO :: เปิดการขาย',
               'SMCO :: เปิดการขาย', 'SMCO :: พิมพ์ใบเสร็จซ้ำ']
titleListIdx = []
matched_string = ""
head_office_str_elmt = '/html/body/div[1]/div[2]/div/div/div/div/div/div/div/div[1]/div[1]/div[2]/div/div[4]/div[2]/div[2]/div[2]/div[1]/div[3]/div[2]'
# สาระ, น่าสนใจ## enumerate() จะคืนค่าให้ตัวแปรloop เป็น object ทำให้เจ้าfor loop ดึงตัวแปรสำหรับ loop ได้มากกว่า 1 ตัว {'ตัวแรกจะได้index', 'ตัวที่สอง จะได้ ค่าvalue'}
for idx, handle in enumerate(driver.window_handles):
    driver.switch_to.window(handle)
    titleListIdx.append(driver.title + "["+str(idx)+"]")
    titleList.append(driver.title)

    value_list.append(driver.current_window_handle)
    title_dict.update({driver.title: driver.current_window_handle})
print("มีไรบ้าง", titleListIdx)
print("จำนวน tabs ตอนเริ่มต้น", len(titleListIdx))

def build_list(list):
    # global result
    result = []
    counter = {}
    for item in list:
        if item in counter:
            counter[item] += 1
            print("counter[item] คือไร: ", counter[item])
            result.append(f"{item}{counter[item]-1}")
        else:
            counter[item] = 1
            result.append(item)
    return result


# ############operation start
# สร้างlistแบบไม่ซ้ำเพื่อทำ unique keyเกบใน title_new
title_new = build_list(titleList)
# เอา unique key รวม กับ value (value ไม่ต้องทำ unique เพราะ unique อยู่แล้ว)
merged_dict = dict(zip(title_new, value_list))


#* The function starts here
def demonic_cp(cp_no):
    driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
    selected_btn_idx = 1
    #*>  element location
    # * >> ปุ่มคูปองด้านนอก ที่ตำแหน่ง [-4] จะเป็นตัวแยก element หรือ ตัวบอกตำแหน่งของ element ว่าเป็นลำดับที่เท่าไหร่ อย่างตัวอย่างนี้เป็น อันที่1
    cp_btn_xpath = '/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div[1]/div/div[2]/div[3]/div[1]/a'
    green_agree_btn_xpath = '/html/body/div[1]/div[2]/div[9]/div/div[1]/div[2]/a'



    items_lsit = driver.find_elements(By.CSS_SELECTOR, '.col-sm-12.panel.panel-default.ng-scope')
    cp_list = driver.find_elements(By.XPATH, '/html/body/div[1]/div[2]/div[9]/div/div[2]/div[3]')

    ordered_items = ['SP2-001713', 'SP2-001714', 'SP2-001715', 'SP2-001716']

    # print("items_lsit", items_lsit)
    for idx, item in enumerate(ordered_items):
        print("มาถึงนี่ไหม")
        for idx2, div in enumerate(items_lsit):
            print("รอบ",idx2)
            # time.sleep(0.55)
            try:
                is_found = div.text.find(item)
            except:
                pass
            li_position = idx+1
            if is_found != -1:
                print("เจอที่ ", li_position)
                print("is_found: ",is_found)
                cp_btn_xpath = f'/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div[{li_position}]/div/div[2]/div[3]/div[1]/a'
                driver.find_element(By.XPATH, cp_btn_xpath).click()
                
                #* เลือก cp เป้าหมาย
                selected_btn = f'/html/body/div[1]/div[2]/div[9]/div/div[2]/div[3]/div[{selected_btn_idx}]/div[1]/button'
                driver.find_element(By.XPATH, selected_btn).click()
                
                driver.find_element(By.XPATH, green_agree_btn_xpath).click()
                # time.sleep(0.55)
                continue
                # print(div.text)
            else :
                print("ไม่เจอ", item, "นะ")
                pass

    print('test_element',cp_list)
    for idx, div in enumerate(cp_list):
        print("CPอันที่",idx," ",div.text)
        
def accel_mode_initialize(account, ):
    dev_account = ["62078", "61651"]

    user_account = "62078"
    is_accel_mode = True


    if user_account in dev_account and is_accel_mode:
        print("Hyper mode Activated")
    else:
        print("Normal mode")
    
def accel_mode(extracted):
    extracted_df = extracted
    driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
    dynamic_inx = 2


    try:
        checkresult = driver.find_element(By.XPATH, f'/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div[{dynamic_inx}]/div/div[1]/div[2]/a')
        print("ผลลัพธ์ถ้าเจอ: ")
        print(checkresult.text)
        print(checkresult)
    except:
        #! มันจะ error ชิพหาย แล้วไม่มี exception ให้ด้วย
        print("ผลลัพธ์ถ้าไม่เจอ: ")
        print("Error !!!")
    driver.quit()
    
def operation_start():
    #* switch to the right page
    driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
    
    #* check ก่อนว่าใส่ชื่อลูกค้ายัง
    cus_name_input_element = driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[6]/form/div/span/span[1]/span/span[1]')
    cus_name_title_attribute = cus_name_input_element.get_attribute("title")
    x = re.search("^C[0-9]+", cus_name_title_attribute)
    try:
        is_name_empty = x.group()
    except:
        print("ไม่มีชื่อลูกค้า")
        return
    
    if is_name_empty:
        #* arguments
        kit_sku = "KCU2-000774"
        target_data = {'code': 'KCU2-000774',
                        'set': '774/001',
                        'cu2': 'U3M0T25304337',
                        'cr4': '601-7D98-280B2401002170',
                        'cr6': 'O710071409',
                        'me1_1': 'A4TNT403X3VI81',
                        'me1_2': 'A4TNT403X3VI81',
                        'ms6': '23505U804947',
                        'ts9': 'A2TXF402502RHW',
                        'ts8': 'A0HJD3433097NM',
                        'ac0': '222823179958',
                        'me1': 'A4TNT403X3VI81 A4TNT403X3VI81'}
        
        #* sku input zone
        sku_input = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[1]/from/div/div/div[1]/div[1]/span/input")
        sku_input.clear()
        sku_input.send_keys(kit_sku)
        sku_input.send_keys(Keys.ENTER)
        
        #* Make sure if an sn btn element appear
        time.sleep(0.25)
        while True:
            try:
                driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div[1]/div[1]/a")
                break
            except:
                continue
            
        
        #* sn fill
        sn_sequence_list = create_sn_fill_sequence()
        #* SN Button Pattern Dir  
        #* /html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div[1]/div[{dom_idx}]/a
        #* Example ref
        #* /html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div[1]/div[1]/a ปุ่ม sn 1
        #* /html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div[1]/div[2]/a ปุ่ม sn 2
        
        
        for i, sku in enumerate(sn_sequence_list):
            dom_idx = i+1
            sn_btn_dir = f"/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[2]/div/div/div[1]/div[2]/div[1]/div[{dom_idx}]/a"
            sn_btn_elmt = driver.find_element(By.XPATH, sn_btn_dir)
            sn_btn_elmt.click()
            
            #* SN input in the pop-up
            while True:
                try:
                    driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[2]/div[2]/div[7]/div/div/div[2]/form/div/div[1]/div/input')
                    break
                except:
                    continue
        
            driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[2]/div[2]/div[7]/div/div/div[2]/form/div/div[1]/div/input').clear()
            driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[2]/div[2]/div[7]/div/div/div[2]/form/div/div[1]/div/input').send_keys(target_data[sku])
            
            #*submit 
            submit_btn = driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[2]/div[2]/div[7]/div/div/div[2]/div[2]/a[1]')
            #* ปุ่มมันกระพริบมันมีช่วงที่ click ได้และไม่ได้ ต้องใช้ while มารัวให้มัน
            for i in range(2):
                while True:
                    try:
                        submit_btn.click()
                        break
                    except: 
                        continue
    else:
        print("ไม่มีชื่อลูกค้า")
        return    
                
        
    
def create_sn_fill_sequence():
    # test case ที่ดีต้องดูหลายๆค่า เพราะแต่ละ sku มีลำดับไม่เหมือนกันฉะนั้นต้องลองเทสสองเคสนี้ KCU2-000781, KCU2-000777
    #* Argument ต้องรับ
    target = "KCU2-000774"
    driver.switch_to.window(merged_dict['SMCO :: เปิดการขาย'])
    # sku_kit_list = driver.find_elements(By.CSS_SELECTOR, "div.col-sm-9.col-xs-8")
    print("มีเซ็ตไรบ้าง")
    try:
        kit_name_target = driver.find_element(By.XPATH, f"//a[contains(., '{target}')]")
    except:
        print(f"หา ชุดkit {target} ไม่เจอ")
        return
    kit_items_target_elmt = kit_name_target.find_element(By.XPATH, "../div[1]")
    kit_items_list = kit_items_target_elmt.find_elements(By.CLASS_NAME, "ng-binding")
    # print(f"element by kit name: {kit_name_target}, kit name: {kit_name_target.text}")
    # print(f"kit_items_target: {kit_items_target_elmt}")
    # print(f"kit_items_list: {kit_items_list}")
    
    #* ดึง element ทั้งหมด
    prog = re.compile(r"\w{2}\d\-\d{6}")
    
    sku_type_list = []
    for i, item in enumerate(kit_items_list):
        try:
            result = f"{prog.match(item.text).group()}"
            sku_type_list.append(str(result[0:3].lower()))
            # print(result[0:3])
            
        except:
            continue
    print("gotcha: ", sku_type_list)
    return sku_type_list
    
    #* forloop นี้ จะบอกจำนวน sku เฉยๆ
    # for i, element in enumerate(sku_kit_list):
    #     print(f"kit ชุดที่ {i+1}")
    # print(sku_kit_list)
    

    
#* Test sonicblow #######################################
# demonic_cp(2)

#* Test accelmode ########################################
# data = {}
# user_account = "62078"
# is_accel_mode = True
# accel_mode(data)

#* Test auto sn ############################################
# fill_order = create_sn_fill_sequence()
operation_start()

# selected_btn = f'/html/body/div[1]/div[2]/div[9]/div/div[2]/div[3]/div[{selected_btn_idx}]/div[1]/button'
# driver.find_element(By.XPATH, selected_btn).click()



handles  ['2D4EC43898A312F81D507020F76393D7', 'EA4F89D70A7CCDD223680E2A7F054A51', 'A028C069FD21027595182A19274B11C0', '0990C1EC4C386E96C81E4678C23BE6C0', 'D3EDABD20453D6B38A6792514C5C10F0', 'A735EB0412C9AC53BBD9512B657FD7DF', '30838AA7993280A50EE88B64B7396143', 'B23430940FDABF91A19EE1973EB0EA70']
มีไรบ้าง ['DevTools[0]', 'DevTools[1]', 'SMCO :: เปิดการขาย[2]', 'คู่มือสินค้า[3]', 'Seller Center[4]', 'SMCO :: ประวัติการขาย[5]', 'Seller Centre[6]', 'SMCO :: ลูกค้า[7]']
จำนวน tabs ตอนเริ่มต้น 8
counter[item] คือไร:  2
มีเซ็ตไรบ้าง
gotcha:  ['cr4', 'cr6', 'me1', 'ms6', 'ts9', 'ts8', 'ac0', 'cu2']
